In [1]:
from google.colab import files
import shutil
from pathlib import Path

# Upload the file
uploaded = files.upload()

raw_dir = Path('/content/capstone_final_dataset_build/raw')
raw_dir.mkdir(parents=True, exist_ok=True) # Ensure raw directory exists

target_csv_filename = 'NFWBS_PUF_2016_data.csv'

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

  # Always save the uploaded CSV with the standardized name expected by downstream cells
  destination_path = raw_dir / target_csv_filename
  with open(destination_path, 'wb') as f:
    f.write(uploaded[fn])
  print(f'Moved uploaded file to "{destination_path}" (renamed from "{fn}")')

Saving NFWBS_PUF_2016_data.csv to NFWBS_PUF_2016_data.csv
User uploaded file "NFWBS_PUF_2016_data.csv" with length 2992071 bytes
Moved uploaded file to "/content/capstone_final_dataset_build/raw/NFWBS_PUF_2016_data.csv" (renamed from "NFWBS_PUF_2016_data.csv")


In [6]:
from pathlib import Path

raw_dir = Path('/content/capstone_final_dataset_build/raw')
target_csv_filename = 'NFWBS_PUF_2016_data.csv'
destination_path = raw_dir / target_csv_filename

if destination_path.exists():
    print(f'File found: {destination_path}')
else:
    print(f'File NOT found: {destination_path}')
    print('Please ensure you have uploaded the CSV file using the upload cell above.')

File found: /content/capstone_final_dataset_build/raw/NFWBS_PUF_2016_data.csv


In [5]:
# ============================================================
# CFPB CAPSTONE: DATASET BUILD
# ============================================================
# Section 1. Setup
import os, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from scipy.stats import f_oneway

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except Exception:
    XGB_OK = False

print('Section 1 complete')


# Section 2. Load CSV
csv_path = raw/'NFWBS_PUF_2016_data.csv'
df = pd.read_csv(csv_path, low_memory=False)
print('CSV shape:', df.shape)
with open(out/'csv_profile.json','w') as f:
    json.dump({'rows':int(df.shape[0]),'cols':int(df.shape[1]),'columns':df.columns.tolist()}, f, indent=2)

# Section 3. Tutor-requested variable families
groups = {
    'outcome': ['FWBscore'],
    'behavioral': ['SAVEHABIT','SCFHORIZON','PROPPLAN_1','PROPPLAN_2','PROPPLAN_3','PROPPLAN_4','MANAGE1_1','MANAGE1_2','MANAGE1_3','MANAGE1_4','ACT1_1','ACT1_2','FINGOALS','GOALCONF','TRACK','PAYCHECK'],
    'stress_hardship': ['DISTRESS','ABSORBSHOCK','VOLATILITY','ENDSMEET','COVERCOSTS','MATHARDSHIP_1','MATHARDSHIP_2','MATHARDSHIP_3','MATHARDSHIP_4','MATHARDSHIP_5','MATHARDSHIP_6'],
    'major_life_events': ['SHOCKS_1','SHOCKS_2','SHOCKS_3','SHOCKS_4','SHOCKS_5','SHOCKS_6','SHOCKS_7','SHOCKS_8','SHOCKS_9','SHOCKS_10','SHOCKS_11'],
    'family_social_background': ['PAREDUC','FINSOC2_1','FINSOC2_2','FINSOC2_3','FINSOC2_4','FINSOC2_5','FINSOC2_6','FINSOC2_7','EARNERS','PPHHSIZE','LIVINGARRANGEMENT'],
    'psychological_values': ['MATERIALISM_1','MATERIALISM_2','MATERIALISM_3','CONNECT','OUTLOOK_1','OUTLOOK_2'],
    'demographic_knowledge_controls': ['agecat','PPINCIMP','PPEDUC','EMPLOY','PPGENDER','PPETHM','LMscore','KHscore','FSscore','finalwt']
    }

all_vars = []
for g in groups:
    all_vars += groups[g]
    all_vars = list(dict.fromkeys(all_vars))
    existing = [c for c in all_vars if c in df.columns]
    missing = [c for c in all_vars if c not in df.columns]

group_map = {}
for g, vars_ in groups.items():
    for v in vars_:
        if v in existing:
            group_map[v] = g

audit = []
for v in existing:
    g = group_map[v]
    rq = 'RQ1,RQ2,RQ3,RQ4' if g in ['behavioral','stress_hardship','psychological_values'] else 'RQ1,RQ3,RQ4'
    if g=='outcome': rq='RQ1,RQ3,RQ4'
    if g=='family_social_background': rq='RQ3,RQ4'
    priority = 'critical' if g=='outcome' else ('high' if g in ['behavioral','stress_hardship','psychological_values','demographic_knowledge_controls'] else 'medium')
    audit.append({
            'variable': v,
            'group': g,
            'criterion_theoretical_relevance': 'Yes',
            'criterion_direct_link_to_rq': rq,
            'criterion_data_quality_variation': 'pending quality check',
            'criterion_non_redundancy': 'checked by variable-family design',
            'priority': priority
            })

audit_df = pd.DataFrame(audit)
print('Section 3 complete. Existing:', len(existing), 'Missing requested:', len(missing))
if missing:
    print('Missing vars (not in official CSV):', missing)

# Section 4. Special codes -> missing
work = df[existing].copy()
special_codes = {-5,-4,-3,-2,-1,98,99}
for c in work.columns:
    if pd.api.types.is_numeric_dtype(work[c]):
        work[c] = work[c].replace(list(special_codes), np.nan)

# Section 5. Quality + keep/drop rule
quality = pd.DataFrame({
    'variable': work.columns,
    'group': [group_map[v] for v in work.columns],
    'missing_n': work.isna().sum().values,
    'missing_pct': (work.isna().mean().values*100).round(2),
    'n_unique': [work[c].nunique(dropna=True) for c in work.columns],
    })

high_priority = set(audit_df[audit_df.priority.isin(['critical','high'])]['variable'])
rules = []
for _, r in quality.iterrows():
    v, miss, nu = r['variable'], r['missing_pct'], r['n_unique']
    if v=='FWBscore':
        d, reason = 'keep','outcome'
    elif nu<=1:
        d, reason = 'drop','no variation'
    elif miss<=15:
        d, reason = 'keep','missing <= 15%'
    elif v in high_priority and miss<=40:
        d, reason = 'keep_with_imputation','high-priority with moderate missingness'
    else:
        d, reason = 'drop','high missingness and lower priority'
    rules.append({'variable':v,'decision':d,'reason':reason})

rule_df = pd.DataFrame(rules)
quality = quality.merge(rule_df, on='variable', how='left')
retained = quality[quality.decision.isin(['keep','keep_with_imputation'])]['variable'].tolist()

qmap = quality.set_index('variable')[['missing_pct','n_unique','decision','reason']].to_dict('index')
audit_df['criterion_data_quality_variation'] = audit_df['variable'].apply(lambda v: f"missing={qmap[v]['missing_pct']}%; unique={int(qmap[v]['n_unique'])}; {qmap[v]['decision']} ({qmap[v]['reason']})")

Section 1 complete
CSV shape: (6394, 217)
Section 3 complete. Existing: 64 Missing requested: 2
Missing vars (not in official CSV): ['TRACK', 'PAYCHECK']


In [8]:
# =====================================
# CFPB CAPSTONE: FINAL DATASET BUILD
# =====================================

import json
import hashlib
import importlib.util
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, log_loss, accuracy_score, precision_score, recall_score, f1_score,
    silhouette_score, calinski_harabasz_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.stats import f_oneway

# Optional XGBoost
XGB_OK = importlib.util.find_spec("xgboost") is not None
if XGB_OK:
    from xgboost import XGBClassifier

# -----------------------------
# 1) Paths
# -----------------------------
base = Path("/content/capstone_final_dataset_build")
raw = base / "raw"
out = base / "outputs"
raw.mkdir(parents=True, exist_ok=True)
out.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2) Official sources
# -----------------------------
sources = {
    "csv": ("NFWBS_PUF_2016_data.csv", "https://files.consumerfinance.gov/f/documents/NFWBS_PUF_2016_data.csv"),
    "codebook_pdf": ("cfpb_nfwbs-puf-codebook.pdf", "https://files.consumerfinance.gov/f/documents/cfpb_nfwbs-puf-codebook.pdf"),
    "user_guide_pdf": ("cfpb_nfwbs-puf-user-guide.pdf", "https://files.consumerfinance.gov/f/documents/cfpb_nfwbs-puf-user-guide.pdf"),
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def download_with_retry(url: str, dest: Path):
    headers = {"User-Agent": "Mozilla/5.0 (Capstone-Data-Build)"}
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()
    dest.write_bytes(r.content)

verification_rows = []

for key, (name, url) in sources.items():
    p = raw / name
    status = "local_existing"
    err = ""
    if not p.exists() or p.stat().st_size == 0:
        try:
            download_with_retry(url, p)
            status = "downloaded"
        except Exception as e:
            status = "download_failed"
            err = str(e)

    verification_rows.append({
        "file_key": key,
        "file_name": name,
        "official_url": url,
        "local_path": str(p),
        "exists": p.exists(),
        "size_bytes": int(p.stat().st_size) if p.exists() else 0,
        "sha256": sha256_file(p) if p.exists() else "",
        "status": status,
        "error_if_any": err
    })

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(out / "official_source_verification.csv", index=False)

# CSV must exist
csv_path = raw / "NFWBS_PUF_2016_data.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        "CSV not found at /content/capstone_final_dataset_build/raw/NFWBS_PUF_2016_data.csv\n"
        "Upload it manually if needed, then re-run this cell."
    )

# -----------------------------
# 3) Load dataset
# -----------------------------
df = pd.read_csv(csv_path, low_memory=False)

with open(out / "csv_profile.json", "w") as f:
    json.dump(
        {"rows": int(df.shape[0]), "cols": int(df.shape[1]), "columns": df.columns.tolist()},
        f, indent=2
    )

# -----------------------------
# 4) Tutor-requested variable groups
# -----------------------------
groups = {
    "outcome": ["FWBscore"],
    "behavioral": [
        "SAVEHABIT", "SCFHORIZON",
        "PROPPLAN_1", "PROPPLAN_2", "PROPPLAN_3", "PROPPLAN_4",
        "MANAGE1_1", "MANAGE1_2", "MANAGE1_3", "MANAGE1_4",
        "ACT1_1", "ACT1_2", "FINGOALS", "GOALCONF",
        "TRACK", "PAYCHECK"
    ],
    "stress_hardship": [
        "DISTRESS", "ABSORBSHOCK", "VOLATILITY", "ENDSMEET", "COVERCOSTS",
        "MATHARDSHIP_1", "MATHARDSHIP_2", "MATHARDSHIP_3",
        "MATHARDSHIP_4", "MATHARDSHIP_5", "MATHARDSHIP_6"
    ],
    "major_life_events": [
        "SHOCKS_1", "SHOCKS_2", "SHOCKS_3", "SHOCKS_4", "SHOCKS_5",
        "SHOCKS_6", "SHOCKS_7", "SHOCKS_8", "SHOCKS_9", "SHOCKS_10", "SHOCKS_11"
    ],
    "family_social_background": [
        "PAREDUC", "FINSOC2_1", "FINSOC2_2", "FINSOC2_3", "FINSOC2_4",
        "FINSOC2_5", "FINSOC2_6", "FINSOC2_7", "EARNERS", "PPHHSIZE", "LIVINGARRANGEMENT"
    ],
    "psychological_values": [
        "MATERIALISM_1", "MATERIALISM_2", "MATERIALISM_3", "CONNECT", "OUTLOOK_1", "OUTLOOK_2"
    ],
    "demographic_knowledge_controls": [
        "agecat", "PPINCIMP", "PPEDUC", "EMPLOY", "PPGENDER", "PPETHM",
        "LMscore", "KHscore", "FSscore", "finalwt"
    ]
}

all_vars = list(dict.fromkeys([v for arr in groups.values() for v in arr]))
existing = [c for c in all_vars if c in df.columns]
missing = [c for c in all_vars if c not in df.columns]

group_map = {v: g for g, arr in groups.items() for v in arr if v in existing}

audit_rows = []
for v in existing:
    g = group_map[v]
    if g in ["behavioral", "stress_hardship", "psychological_values"]:
        rq = "RQ1,RQ2,RQ3,RQ4"
    elif g == "family_social_background":
        rq = "RQ3,RQ4"
    else:
        rq = "RQ1,RQ3,RQ4"

    if g == "outcome":
        priority = "critical"
    elif g in ["behavioral", "stress_hardship", "psychological_values", "demographic_knowledge_controls"]:
        priority = "high"
    else:
        priority = "medium"

    audit_rows.append({
        "variable": v,
        "group": g,
        "criterion_theoretical_relevance": "Yes",
        "criterion_direct_link_to_rq": rq,
        "criterion_data_quality_variation": "pending quality check",
        "criterion_non_redundancy": "checked by variable-family design",
        "priority": priority
    })

audit_df = pd.DataFrame(audit_rows)

# -----------------------------
# 5) Special code handling
# -----------------------------
work = df[existing].copy()
special_codes = {-5, -4, -3, -2, -1, 98, 99}
for c in work.columns:
    if pd.api.types.is_numeric_dtype(work[c]):
        work[c] = work[c].replace(list(special_codes), np.nan)

# -----------------------------
# 6) Quality report + keep/drop rules
# -----------------------------
quality = pd.DataFrame({
    "variable": work.columns,
    "group": [group_map[v] for v in work.columns],
    "missing_n": work.isna().sum().values,
    "missing_pct": (work.isna().mean().values * 100).round(2),
    "n_unique": [work[c].nunique(dropna=True) for c in work.columns],
})

high_priority = set(audit_df[audit_df["priority"].isin(["critical", "high"])]["variable"])

rules = []
for _, r in quality.iterrows():
    v = r["variable"]
    miss = r["missing_pct"]
    nu = r["n_unique"]

    if v == "FWBscore":
        d, reason = "keep", "outcome"
    elif nu <= 1:
        d, reason = "drop", "no variation"
    elif miss <= 15:
        d, reason = "keep", "missing <= 15%"
    elif v in high_priority and miss <= 40:
        d, reason = "keep_with_imputation", "high-priority with moderate missingness"
    else:
        d, reason = "drop", "high missingness and lower priority"

    rules.append({"variable": v, "decision": d, "reason": reason})

rule_df = pd.DataFrame(rules)
quality = quality.merge(rule_df, on="variable", how="left")
retained = quality[quality["decision"].isin(["keep", "keep_with_imputation"])]["variable"].tolist()

qmap = quality.set_index("variable")[["missing_pct", "n_unique", "decision", "reason"]].to_dict("index")
audit_df["criterion_data_quality_variation"] = audit_df["variable"].apply(
    lambda v: f"missing={qmap[v]['missing_pct']}%; unique={int(qmap[v]['n_unique'])}; {qmap[v]['decision']} ({qmap[v]['reason']})"
)

# -----------------------------
# 7) Row filters + imputation
# -----------------------------
clean = work[retained].copy()
start_n = len(clean)

clean = clean[clean["FWBscore"].notna()].copy()
removed_outcome = start_n - len(clean)

core_candidates = [
    "SAVEHABIT", "SCFHORIZON", "ACT1_1", "ACT1_2", "FINGOALS", "GOALCONF",
    "DISTRESS", "ABSORBSHOCK", "VOLATILITY", "ENDSMEET",
    "agecat", "PPINCIMP", "PPEDUC", "EMPLOY", "LMscore", "KHscore", "FSscore"
]
core = [c for c in core_candidates if c in clean.columns]

if core:
    clean["row_missing_core_pct"] = clean[core].isna().mean(axis=1)
    before_core = len(clean)
    clean = clean[clean["row_missing_core_pct"] <= 0.30].copy()
    removed_core = before_core - len(clean)
else:
    clean["row_missing_core_pct"] = 0.0
    removed_core = 0

impute_rows = []
for c in clean.columns:
    if c in ["FWBscore", "row_missing_core_pct"]:
        continue

    m = int(clean[c].isna().sum())
    if m == 0:
        continue

    vals = clean[c].dropna()
    uniq = set(vals.unique().tolist())

    if uniq and uniq.issubset({0, 1}):
        fill = vals.mode().iloc[0]
        method = "mode_binary"
    elif pd.api.types.is_numeric_dtype(clean[c]):
        fill = vals.median()
        method = "median_numeric"
    else:
        fill = vals.mode().iloc[0]
        method = "mode_categorical"

    clean[c] = clean[c].fillna(fill)
    impute_rows.append({
        "variable": c,
        "method": method,
        "fill_value": float(fill) if isinstance(fill, (int, float, np.number)) else str(fill),
        "missing_filled_n": m
    })

impute_df = pd.DataFrame(impute_rows)

# -----------------------------
# 8) Derived variables
# -----------------------------
hardship_cols = [c for c in ["MATHARDSHIP_1","MATHARDSHIP_2","MATHARDSHIP_3","MATHARDSHIP_4","MATHARDSHIP_5","MATHARDSHIP_6"] if c in clean.columns]
if hardship_cols:
    hb = pd.DataFrame(index=clean.index)
    for c in hardship_cols:
        u = set(clean[c].dropna().unique().tolist())
        if u.issubset({0, 1}):
            hb[c] = clean[c].astype(int)
        elif u.issubset({1, 2}):
            hb[c] = (clean[c] == 1).astype(int)
        else:
            hb[c] = (clean[c] >= 1).astype(int)
    clean["HARDSHIP_TOTAL"] = hb.sum(axis=1)
else:
    clean["HARDSHIP_TOTAL"] = np.nan

clean["FWB_high_54"] = (clean["FWBscore"] >= 54).astype(int)
clean["FWB_high_56"] = (clean["FWBscore"] >= 56).astype(int)
clean["FWBcat"] = np.select(
    [clean["FWBscore"] < 44, clean["FWBscore"] <= 61],
    ["Low", "Medium"],
    default="High"
)

# -----------------------------
# 9) Weighted checks
# -----------------------------
if "finalwt" in clean.columns:
    wt = clean["finalwt"].fillna(1.0).astype(float)
else:
    wt = pd.Series(np.ones(len(clean)), index=clean.index)

weighted = {
    "unweighted_mean_FWBscore": float(clean["FWBscore"].mean()),
    "weighted_mean_FWBscore": float(np.average(clean["FWBscore"], weights=wt)),
    "unweighted_high54_rate": float(clean["FWB_high_54"].mean()),
    "weighted_high54_rate": float(np.average(clean["FWB_high_54"], weights=wt)),
    "unweighted_high56_rate": float(clean["FWB_high_56"].mean()),
    "weighted_high56_rate": float(np.average(clean["FWB_high_56"], weights=wt)),
}

# -----------------------------
# 10) Model sanity checks
# -----------------------------
def cls_metrics(y_true, prob):
    pred = (prob >= 0.5).astype(int)
    return {
        "auc_roc": float(roc_auc_score(y_true, prob)),
        "log_loss": float(log_loss(y_true, prob)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
    }

def run_models(target_col):
    y = clean[target_col].astype(int)

    baseline = [c for c in ["agecat","PPINCIMP","PPEDUC"] if c in clean.columns]
    model_a = [c for c in ["agecat","PPINCIMP","PPEDUC","EMPLOY","LMscore","KHscore","FSscore"] if c in clean.columns]

    behavior_vars = []
    for g in ["behavioral","stress_hardship","major_life_events","family_social_background","psychological_values"]:
        behavior_vars += [v for v in groups[g] if v in clean.columns]
    behavior_vars = list(dict.fromkeys(behavior_vars))
    model_b = list(dict.fromkeys(model_a + behavior_vars))

    def fit_block(features):
        if len(features) < 2:
            return {"status": "skipped_not_enough_features"}

        X = clean[features].astype(float)
        if y.nunique() < 2:
            return {"status": "skipped_one_class_target"}

        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )

        outm = {}

        lr = LogisticRegression(max_iter=3000)
        lr.fit(Xtr, ytr)
        outm["logistic_regression"] = cls_metrics(yte, lr.predict_proba(Xte)[:, 1])

        rf = RandomForestClassifier(n_estimators=350, min_samples_leaf=2, random_state=42, n_jobs=-1)
        rf.fit(Xtr, ytr)
        outm["random_forest"] = cls_metrics(yte, rf.predict_proba(Xte)[:, 1])

        if XGB_OK:
            xgb = XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.85,
                colsample_bytree=0.85,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
            xgb.fit(Xtr, ytr)
            outm["xgboost"] = cls_metrics(yte, xgb.predict_proba(Xte)[:, 1])
        else:
            outm["xgboost"] = {"status": "not_available"}

        return outm

    return {
        "target": target_col,
        "baseline_model": fit_block(baseline),
        "model_A_demog_knowledge": fit_block(model_a),
        "model_B_plus_behavior": fit_block(model_b)
    }

model_54 = run_models("FWB_high_54")
model_56 = run_models("FWB_high_56")

# -----------------------------
# 11) Clustering
# -----------------------------
cluster_features = []
for g in ["behavioral","stress_hardship","major_life_events","psychological_values"]:
    cluster_features += [v for v in groups[g] if v in clean.columns]
cluster_features = list(dict.fromkeys(cluster_features))

cluster_summary = {"cluster_feature_count": len(cluster_features)}

if len(cluster_features) >= 3:
    Xs = StandardScaler().fit_transform(clean[cluster_features].astype(float))

    best = None
    k_grid = []
    for k in [3, 4, 5, 6]:
        km = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = km.fit_predict(Xs)
        sil = float(silhouette_score(Xs, labels, sample_size=min(3000, len(Xs)), random_state=42))
        ch = float(calinski_harabasz_score(Xs, labels))
        k_grid.append({"k": k, "silhouette": sil, "calinski_harabasz": ch})
        if (best is None) or (sil > best["silhouette"]):
            best = {"k": k, "silhouette": sil, "calinski_harabasz": ch, "labels": labels}

    clean["behavior_cluster"] = best["labels"]
    by_cluster = [clean.loc[clean["behavior_cluster"] == k, "FWBscore"].values for k in sorted(clean["behavior_cluster"].unique())]
    an = f_oneway(*by_cluster)
    means = clean.groupby("behavior_cluster")["FWBscore"].agg(["count","mean","std"]).reset_index()

    cluster_summary.update({
        "k_grid": k_grid,
        "best_k": int(best["k"]),
        "best_silhouette": float(best["silhouette"]),
        "best_calinski_harabasz": float(best["calinski_harabasz"]),
        "anova_f_stat": float(an.statistic),
        "anova_p_value": float(an.pvalue),
        "cluster_means_fwbscore": means.to_dict(orient="records")
    })
else:
    clean["behavior_cluster"] = -1
    cluster_summary["status"] = "skipped_not_enough_features"

# -----------------------------
# 12) Save outputs
# -----------------------------
clean.to_csv(out / "final_dataset_capstone_v2.csv", index=False)
quality.to_csv(out / "variable_missingness_quality_report.csv", index=False)
audit_df.to_csv(out / "variable_selection_audit.csv", index=False)
impute_df.to_csv(out / "imputation_log.csv", index=False)

pd.DataFrame([
    {"step":"start_rows", "rows":start_n},
    {"step":"drop_missing_outcome", "rows_removed":removed_outcome},
    {"step":"drop_row_missing_core_gt_30pct", "rows_removed":removed_core},
    {"step":"final_rows", "rows":len(clean)}
]).to_csv(out / "row_filter_log.csv", index=False)

with open(out / "model_sanity_check.json", "w") as f:
    json.dump({"threshold_54": model_54, "threshold_56": model_56}, f, indent=2)

with open(out / "cluster_summary.json", "w") as f:
    json.dump(cluster_summary, f, indent=2)

summary = {
    "raw_shape": {"rows": int(df.shape[0]), "cols": int(df.shape[1])},
    "final_shape": {"rows": int(clean.shape[0]), "cols": int(clean.shape[1])},
    "retained_variable_count": int(len(retained)),
    "requested_missing_in_csv_count": int(len(missing)),
    "missing_requested_vars": missing,
    "weighted_summary": weighted,
    "files": {
        "final_dataset": str(out / "final_dataset_capstone_v2.csv"),
        "source_verification": str(out / "official_source_verification.csv"),
        "quality_report": str(out / "variable_missingness_quality_report.csv"),
        "selection_audit": str(out / "variable_selection_audit.csv"),
        "imputation_log": str(out / "imputation_log.csv"),
        "row_filter_log": str(out / "row_filter_log.csv"),
        "model_sanity_check": str(out / "model_sanity_check.json"),
        "cluster_summary": str(out / "cluster_summary.json"),
    }
}
with open(out / "final_dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive("/content/capstone_final_dataset_build_outputs", "zip", out)

print("ALL DONE")
print("Output folder:", out)
print("Zip file:", zip_path)
print("Raw shape:", df.shape, "Final shape:", clean.shape)
print("Missing requested vars:", missing)
print("Weighted mean FWBscore:", round(weighted["weighted_mean_FWBscore"], 4))
print("Weighted high54 rate:", round(weighted["weighted_high54_rate"], 4))
print("Weighted high56 rate:", round(weighted["weighted_high56_rate"], 4))


ALL DONE
Output folder: /content/capstone_final_dataset_build/outputs
Zip file: /content/capstone_final_dataset_build_outputs.zip
Raw shape: (6394, 217) Final shape: (6374, 70)
Missing requested vars: ['TRACK', 'PAYCHECK']
Weighted mean FWBscore: 54.2403
Weighted high54 rate: 0.5194
Weighted high56 rate: 0.4604
